In [1]:
import os
import shutil
import time
from pathlib import Path

import cv2
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix, precision_recall_curve

from anomalib.data import Folder
from anomalib.engine import Engine
from anomalib.models import Patchcore
from lightning.pytorch.callbacks import TQDMProgressBar

from anomalib.data import Folder
from anomalib.data.utils.split import ValSplitMode, TestSplitMode
from anomalib.models import Patchcore
from anomalib.engine import Engine

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

PyTorch version : 2.11.0+cpu
CUDA available  : False


In [5]:
# ── Dataset ────────────────────────────────────────────────────────────

DATASET_ROOT = r"C:\Users\Inga\miniconda3\envs\capstone\capstone_xray\data\xray\anomalib_data\Machine_S"
DATASET_NAME = ""

# ── Model ──────────────────────────────────────────────────────────────
BACKBONE       = "mobilenetv3_small_100"
LAYERS         = ["blocks.3", "blocks.5"]
NUM_NEIGHBORS  = 9
CORESET_RATIO = 0.01

# ── DataLoader ─────────────────────────────────────────────────────────
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE  = 1
NUM_WORKERS      = 0

# ── Output ─────────────────────────────────────────────────────────────
#SAVE_DIR = Path("results/final_pipeline")
#SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Benchmarking ───────────────────────────────────────────────────────
# All latency measurements are performed exclusively on CPU.
NUM_RUNS           = 10
BENCH_WARMUP_RUNS  = 20    # warm-up passes (discarded)
BENCH_MEASURE_RUNS = 200   # timed passes
BENCH_IMAGE_SIZE   = (1, 3, 256, 256)  # (N, C, H, W) — must match training resolution


accelerator_type = "gpu" if torch.cuda.is_available() else "cpu"
engine = Engine(
    accelerator=accelerator_type,
    devices=1,
    callbacks=[TQDMProgressBar(refresh_rate=0)],
)

print(f"Model      : PatchCore  |  Backbone: {BACKBONE}")
print(f"Layers     : {LAYERS}")
print(f"Neighbors  : {NUM_NEIGHBORS}")
print(f"Accelerator: {accelerator_type}")

print("Configuration loaded.")

Model      : PatchCore  |  Backbone: mobilenetv3_small_100
Layers     : ['blocks.3', 'blocks.5']
Neighbors  : 9
Accelerator: cpu
Configuration loaded.


In [6]:
import os
from pathlib import Path

print('Creating Anomalib data module...')

# --- Image Counting Logic ---
# Construct the paths using the same variables your Folder module uses
train_good_path = Path(DATASET_ROOT) / DATASET_NAME / "train/good"
test_bad_path = Path(DATASET_ROOT) / DATASET_NAME / "test/bad"
test_good_path = Path(DATASET_ROOT) / DATASET_NAME / "test/good"

# Define standard image extensions to filter out non-image files if any exist
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.TIFF')

def count_images(path):
    if not path.exists():
        return 0
    # Counts files that match the image extensions
    return sum(1 for f in path.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions)

print("--- Dataset Image Counts ---")
print(f"Train Normal ('train/good'): {count_images(train_good_path)} images")
print(f"Test Abnormal ('test/bad'):  {count_images(test_bad_path)} images")
print(f"Test Normal ('test/good'):   {count_images(test_good_path)} images")
print("----------------------------\n")


Creating Anomalib data module...
--- Dataset Image Counts ---
Train Normal ('train/good'): 488 images
Test Abnormal ('test/bad'):  255 images
Test Normal ('test/good'):   123 images
----------------------------



In [17]:
def run_single_experiment(run_num, base_save_dir, dataset_root, backbone, layers, num_neighbors, 
                          coreset_sampling_ratio, train_batch_size, eval_batch_size, num_workers, 
                          bench_warmup_runs, bench_measure_runs, bench_image_size):
    
    # Setup unique save directory
    run_save_dir = base_save_dir / f"run_{run_num}"
    if run_save_dir.exists():
        shutil.rmtree(run_save_dir)
    run_save_dir.mkdir(parents=True, exist_ok=True)
    
    # DataModule - Updated to use function parameters
    datamodule = Folder(
        root=dataset_root,
        name="",
        normal_dir="train/good",
        abnormal_dir="test/bad",
        normal_test_dir="test/good",
        train_batch_size=train_batch_size,
        eval_batch_size=eval_batch_size,
        num_workers=num_workers,
        val_split_mode=ValSplitMode.SAME_AS_TEST,
    )
    
    datamodule.setup()
    print(len(datamodule.val_data))
    print(len(datamodule.test_data))
    
    # Model & Engine - FIX: Added `layers=layers` and updated to use function parameters
    model = Patchcore(
        backbone=backbone,
        layers=layers, # <--- THIS WAS THE MISSING LINE CAUSING THE CRASH
        num_neighbors=num_neighbors,
        pre_trained=True,
        coreset_sampling_ratio=coreset_sampling_ratio
    )
    
    engine = Engine(accelerator="auto", devices=1, default_root_dir=str(run_save_dir),
                    callbacks=[TQDMProgressBar(refresh_rate=0)])
    
    # Train & Test
    print("\n--- Diagnostic: Inspecting Test DataLoader ---")
    test_loader = datamodule.test_dataloader()
    num_test_batches = len(test_loader)
    # If batch_size=1, this is equal to the total number of images
    total_samples = num_test_batches * test_loader.batch_size 
    print(f"Test DataLoader configuration: {num_test_batches} batches of size {test_loader.batch_size}")
    print(f"Total images scheduled for testing: {total_samples}")
    print("----------------------------------------------\n")

    
    engine.fit(model=model, datamodule=datamodule)
    test_results = engine.test(model=model, datamodule=datamodule)
    metrics = test_results[0]
    
    # Threshold Selection
    preds = engine.predict(model=model, datamodule=datamodule)
    y_true = np.concatenate([getattr(b, "label", b.gt_label).cpu().numpy() for b in preds])
    y_scores = np.concatenate([getattr(b, "pred_scores", b.pred_score).cpu().numpy() for b in preds])
    
    precision, recall, thresholds = precision_recall_curve(y_true, y_scores)
    f1 = 2 * recall * precision / (recall + precision + 1e-10)
    best_thresh = thresholds[np.argmax(f1)]
    
    # Confusion Matrix
    y_pred = (y_scores >= best_thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Latency Benchmarking
    model_cpu = model.eval().to("cpu")
    dummy = torch.randn(*bench_image_size)
    with torch.no_grad():
        for _ in range(bench_warmup_runs): model_cpu(dummy)
        latencies = []
        for _ in range(bench_measure_runs):
            start = time.perf_counter()
            model_cpu(dummy)
            latencies.append((time.perf_counter() - start) * 1000)
            
    return {
        "Run": run_num, "Backbone": backbone, "Coreset": coreset_sampling_ratio,
        "AUROC": metrics.get('image_AUROC', 0), "F1": np.max(f1), "Threshold": best_thresh,
        "FP": fp, "FN": fn, "FP Rate": (fp / (fp + tn + 1e-10)) * 100,
        "FN Rate": (fn / (fn + tp + 1e-10)) * 100, "Latency (ms)": np.mean(latencies),
        "Test Images": len(y_true)
    }

In [10]:
all_results = []
base_save_dir = Path(r"C:\Users\Inga\miniconda3\envs\capstone\capstone_xray\notebooks\results\MobileNet_experiments")

for i in range(1, NUM_RUNS + 1):
    print(f"Executing Run {i}/{NUM_RUNS}...")
    metrics = run_single_experiment(
        i, base_save_dir, DATASET_ROOT, BACKBONE, LAYERS, NUM_NEIGHBORS, 
        CORESET_RATIO, TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE, NUM_WORKERS,
        BENCH_WARMUP_RUNS, BENCH_MEASURE_RUNS, BENCH_IMAGE_SIZE
    )
    all_results.append(metrics)

# Generate Table & CSV
df = pd.DataFrame(all_results)
df.to_csv(base_save_dir / "all_experiments_summary.csv", index=False)
print(df.to_string(index=False))

Executing Run 1/10...
378
378


Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



--- Diagnostic: Inspecting Test DataLoader ---
Test DataLoader configuration: 378 batches of size 1
Total images scheduled for testing: 378
----------------------------------------------



C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\core\optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │  927 K │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

...

Trainable params: 927 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 927 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 19                                                                                          
Modules in eval mode: 227                                                                                          
Total FLOPs: 0

C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\loops\fit_loop.py:534: Found 227 module(s) in eval mode at the start of tr

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.9920930862426758     │
│       image_F1Score       │    0.9666011929512024     │
└───────────────────────────┴───────────────────────────┘

ckpt_path is not provided. Model weights will not be loaded.
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
C:\Users\Inga\miniconda3\envs\capstone\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


NameError: name 'pd' is not defined

In [19]:
import pandas as pd
all_results.append(metrics)

# Generate Table & CSV
df = pd.DataFrame(all_results)
df.to_csv(base_save_dir / "all_experiments_summary.csv", index=False)
print(df.to_string(index=False))

 Run              Backbone  Coreset    AUROC       F1  Threshold  FP  FN  FP Rate  FN Rate  Latency (ms)
   1 mobilenetv3_small_100     0.01 0.998502 0.992157        0.5   2   2 1.626016 0.784314     58.821176
   2 mobilenetv3_small_100     0.01 0.995919 0.982318        0.5   4   5 3.252033 1.960784     58.200862
   3 mobilenetv3_small_100     0.01 0.996047 0.986355        0.5   5   2 4.065041 0.784314     72.886976
   4 mobilenetv3_small_100     0.01 0.997322 0.980315        0.5   4   6 3.252033 2.352941     59.793365
   5 mobilenetv3_small_100     0.01 0.996652 0.984496        0.5   7   1 5.691057 0.392157     56.501798
   6 mobilenetv3_small_100     0.01 0.995855 0.982318        0.5   4   5 3.252033 1.960784     58.640931
   7 mobilenetv3_small_100     0.01 0.997417 0.984436        0.5   6   2 4.878049 0.784314     57.659749
   8 mobilenetv3_small_100     0.01 0.996047 0.980545        0.5   7   3 5.691057 1.176471     57.387003
   9 mobilenetv3_small_100     0.01 0.995887 0.980545  